# 🧬 Project 5 — Semantic Correspondence
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup
Eseguire per configurare Drive e caricare il modulo demo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b osagie5 https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

# Caricamento motore demo
from utils.demo_utils import launch_stage_demo

## 🔍 1. Stage 1: Backbone Analysis
Analisi zero-shot della Baseline.

In [ ]:
# 1.1 PCK Baseline (Layer 11)
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --results_file /content/drive/MyDrive/AML/Results/baseline.txt

In [ ]:
# 🎮 Demo Baseline (Layer 11 fiso)
launch_stage_demo('DINOv2 Baseline', ckpt_name=None, layer_idx=11)

In [ ]:
# 1.2 Analisi PCK Layer-wise
for layer in [4, 8, 11]:
    print(f'\n--- Testing Layer {layer} ---')
    !python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --layer {layer} --results_file /content/drive/MyDrive/AML/Results/layer_{layer}.txt

In [ ]:
# 🎮 Demo Layer Explorer (Slider)
launch_stage_demo('DINOv2 Layer-wise Explorer', ckpt_name=None, show_layer_slider=True)

## 🚀 2. Stage 2: Fine-Tuning Efficiente (LoRA)
Addestramento modulare.

In [ ]:
# 2.1 Solo LoRA
!python train.py --epochs 5 --curriculum_epochs 0 --num_workers 4 --output_dir ./checkpoints/lora_only --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_only

In [ ]:
# 🎮 Demo LoRA Only
launch_stage_demo('LoRA Only Inference', ckpt_name='lora_only')

In [ ]:
# 2.2 LoRA + Curriculum
!python train.py --epochs 5 --curriculum_epochs 2 --num_workers 4 --output_dir ./checkpoints/lora_curriculum --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_curriculum

## 🎯 3. Stage 3: Raffinamento Adattivo
Ablation dell'Adaptive Windowing.

In [ ]:
CKPT = '/content/drive/MyDrive/AML/Checkpoints/lora_curriculum/best.pth'
!python evaluate.py --checkpoint {CKPT} --results_file /content/drive/MyDrive/AML/Results/s3_with.txt
!python evaluate.py --checkpoint {CKPT} --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/s3_without.txt

## 🌟 4. Stage 4: Valutazione ed Estensioni
Demo finale del miglior modello.

In [ ]:
# 🎮 Demo Finale (LoRA + Curriculum)
launch_stage_demo('Modello Finale', ckpt_name='lora_curriculum')